# LC
### To Do: 
 - Three examples for paper

# コード
## Utility: 
###  - LCファイル名一覧の取得

In [ ]:
import os

def get_lc_files(dir_path_lc= "./data_LC/14d"): 
    filenames= [
        f for f in os.listdir(dir_path_lc) if os.path.isfile(os.path.join(dir_path_lc, f))
    ]
    # print(filenames)
    return filenames

In [ ]:
get_lc_files()

### - 天体名対応辞書作成

In [ ]:
from astropy.io import fits
import numpy as np

def get_dict_sourcenames():
  num_sources = 2300
  hdu=fits.open('/Users/kazuma/Workspace/Torun/Fermi/catalog/gll_psc_v35.fit') # MacBook
  # hdu=fits.open('/home/kazuma/Workspace/Fermi/gll_psc_v35.fit') # Legion
  significances = hdu[1].data['Signif_Avg']
  sources = hdu[1].data['Source_Name']
  sources1 = hdu[1].data['ASSOC1']
  sources2 = hdu[1].data['ASSOC2']
  sources_cls=hdu[1].data['CLASS1']
  #Convert source classes to normal array without empty spaces
  source_classes = np.array([entry.strip() for entry in hdu[1].data['CLASS1']])
  #Filter by source class:
  source_classes_selected = np.array(["bll","fsrq","BLL","FSRQ"],dtype='<U5') #see table 5 in https://arxiv.org/pdf/2201.11184
  element_map = np.isin(source_classes, source_classes_selected)
  significances_blazars= significances[element_map]
  sources_blazars = sources[element_map]
  sources_blazars1 = sources1[element_map] # data['ASSOC1']
  sources_blazars2 = sources2[element_map] # data['ASSOC2']
  sources_blazars_cls = source_classes[element_map]
  #Get index of 20 brightes sources:
  idx = (-significances_blazars).argsort()[:num_sources]
  indices = np.arange(1,num_sources+1)
  # print(f"Number of sources: {len(sources_blazars[idx])}")
  # print(f"Indices of the sources: {indices}")
  #Get the same of the 20 most significant blazars:
  # print(sources_blazars[idx])
  from astropy.table import Table
  # t = Table([sources_blazars[idx],
  #   sources_blazars1[idx],
  #   sources_blazars_cls[idx],
  #   ],names=['4FGL name','assoc name','CLASS','index'])
  
  
  sources_blazars_converted  = sources_blazars.strip().replace(' ','_').lower()
  sources_blazars1_converted = sources_blazars1.strip().replace(' ','_').lower()
  sources_blazars_selected_cls = sources_blazars_cls[idx]
  # The elements of sources_blazars_selected_cls are mixture of upper/lower cases. 
  # We unify them to upper case here.
  sources_blazars_selected_cls_converted = np.array([entry.upper() for entry in sources_blazars_selected_cls])

#   dict_sourcenames = dict(zip(sources_blazars_converted[idx],sources_blazars1_converted[idx]))

  ref_tab_obj = Table([sources_blazars_converted[idx],
  sources_blazars1_converted[idx],
  sources_blazars_selected_cls_converted, #sources_blazars_cls[idx], 
  indices],
  names=['4FGL name','assoc name','CLASS','index'])
# ref_tab_obj[ref_tab_obj['4FGL name'] == '4fgl_j0112.1+2245']
  return ref_tab_obj


In [ ]:
dict_sourcename = get_dict_sourcenames()
# dict_sourcename[0:15]
dict_sourcename[200:215]

# dict_sourcename[dict_sourcename['4FGL name'] == '4fgl_j0112.1+2245']# dict_sourcename[(dict_sourcename['4FGL name'] == '4fgl_j0112.1+2245')]

In [ ]:
# mask = (dict_sourcename['assoc name'] == 'gb6 j1040+0617') 
mask = (dict_sourcename['4FGL name'] == '4fgl_j1040.5+0617') 
# mask = (dict_sourcename['4FGL name'] == '4fgl_j0348.6-2749') 
dict_sourcename[mask]


In [ ]:
mask = (dict_sourcename['assoc name'] == 'txs_0506+056') 
# mask = (dict_sourcename['4FGL name'] == '4fgl_j1040.5+0617') 
dict_sourcename[mask]

In [ ]:
mask = (dict_sourcename['4FGL name'] == '4fgl_j0348.6-2749')
dict_sourcename[mask]
# dict_sourcename[14]
# mask
# dict_sourcename[14]['4FGL name']= '4fgl_j0348.5-2749'
# dict_sourcename[dict_sourcename['4FGL name'] == '4fgl_j0348.5-2749']
# dict_sourcename['4FGL name'] == '4fgl_j0348.5-2749'
# dict_sourcename[dict_sourcename['4FGL name'] == '4fgl_j0348.6-2749']


# LCの描画
## 全LCの描画

In [ ]:
from astropy.table import Table
import matplotlib.pyplot as plt
import os

lc_file_dir = 'data_LC/14d'
file_list = get_lc_files(lc_file_dir) 

for file in file_list:
    if file.endswith('.fits'):
        lc_tab = Table.read(os.path.join(lc_file_dir, file))
        # Add a new column with the average of tmax_mjd and tmin_mjd
        lc_tab.add_column((lc_tab['tmax_mjd'] + lc_tab['tmin_mjd']) / 2., name='t_mjd', index=0)
        # # Save the modified table back to a new fits file
        # lc_tab.write(os.path.join(lc_file_dir, f'modified_{file}'), overwrite=True)
        # fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(15,10))
        sourcename_4fgl = file.split('_')[1] # Extract the 4FGL name from the filename
        sourceinfo = dict_sourcename[dict_sourcename['4FGL name'] == '4fgl_' + sourcename_4fgl]
        print(sourcename_4fgl)
        sourcename_assoc = sourceinfo['assoc name'].data[0]
        sourceindex = sourceinfo['index'].data[0]
        print('4fgl_' + sourcename_4fgl + ':' + str(sourcename_assoc) + ' ' + str(sourceindex))
        # print(sourcename_assoc)
        lc_x=lc_tab['t_mjd']
        lc_y=lc_tab['flux']
        plt.plot(lc_x,lc_y ,marker='o',label=sourcename_assoc)

# lightcurve,= plt.plot(lc_x[fermipresentbin],lc_y[fermipresentbin] ,marker='o',color='red')
# plt.set(xlim=[mjd_min, mjd_max], ylim=[0,0.3e-4])
plt.xlabel('MJD')
plt.ylabel('flux (cm$^{-2}$ s$^{-1}$)')
plt.yscale('log')
plt.ylim(1e-8, 1e-4)
plt.legend()
plt.show()
plt.tight_layout()


## 選択的に描画（明るさ順指定可能）

In [ ]:
from astropy.table import Table
import matplotlib.pyplot as plt
import os

lc_file_dir = 'data_LC/14d'
file_list = get_lc_files(lc_file_dir) 

plot_refsource = True
if plot_refsource:
 # Plot the light curve of the reference source
  dict_sourcename = get_dict_sourcenames()
  # print(dict_sourcename)
  # print(dict_sourcename[0])
  # print(dict_sourcename[0]['4FGL name'])
  # print(dict_sourcename[0]['assoc name'])
  # print(dict_sourcename[0]['index'])
  source = dict_sourcename[0]
  print(source)
  lc_tab = Table.read(os.path.join(lc_file_dir, source['4FGL name'] + '_lightcurve.fits'))
  lc_tab.add_column((lc_tab['tmax_mjd'] + lc_tab['tmin_mjd']) / 2., name='t_mjd', index=0)
  lc_x=lc_tab['t_mjd']
  lc_y=lc_tab['flux']
  plt.plot(lc_x,lc_y ,marker='o',color= 'red', label=source['assoc name'])



for source in dict_sourcename[11:14]:
  print(source)
  lc_tab = Table.read(os.path.join(lc_file_dir, source['4FGL name'] + '_lightcurve.fits'))
  lc_tab.add_column((lc_tab['tmax_mjd'] + lc_tab['tmin_mjd']) / 2., name='t_mjd', index=0)
  lc_x=lc_tab['t_mjd']
  lc_y=lc_tab['flux']
  plt.plot(lc_x,lc_y ,marker='o',label=source['assoc name'])

plt.xlabel('MJD')
plt.ylabel('flux (cm$^{-2}$ s$^{-1}$)')
plt.yscale('log')
plt.ylim(1e-8, 1e-4)
# plt.xlim(57790-14*20-5, 57790+14*20+5)  # Adjust the x-axis limits to show a 14-day window around MJD 57790
plt.legend()
plt.show()
plt.tight_layout()

# 1d SED描画可能性検証



In [ ]:
lc_file_dir = 'data_LC/1d'
file_list = get_lc_files(lc_file_dir) 
print(file_list)
dict_source = get_dict_sourcenames()
dict_source[dict_sourcename['assoc name'] == '3c_454.3']



### - 1d LC ファイル読込
### - SED ecsv ファイル読込
### - 存否検索・列追加
### - 1d LC の描画

In [ ]:
from astropy.table import Table
import matplotlib.pyplot as plt
import os
lc_file_dir = 'data_LC/1d'
source = dict_source[dict_source['assoc name'] == '3c_454.3']
print(source['4FGL name'].data[0])

###########################
### Read the data       ###
###########################
# LC
# lc_filepath  = os.path.join(lc_file_dir, source['4FGL name'].data[0] +'_lightcurve1.fits')
lc_filepath  = os.path.join(lc_file_dir, source['4FGL name'].data[0] +'_lightcurve2.fits')
lc_tab = Table.read(lc_filepath)

# SED
# sed_filepath = 'data/3C454.3_allsed_1d_min11.ecsv'
sed_filepath = 'data/3C454.3_allsed_1d_2_min11.ecsv'
sed_tab_0 = Table.read(sed_filepath)
nonzero_mask = (sed_tab_0['e2dnde'] > sed_tab_0['e2dnde_err'])
sed_tab = sed_tab_0[nonzero_mask]

lc_tab.add_column((lc_tab['tmax_mjd'] + lc_tab['tmin_mjd']) / 2., name='t_mjd', index=0)
lc_x=lc_tab['t_mjd']
lc_y=lc_tab['flux']
# plt.plot(lc_x,lc_y ,marker='o',color= 'blue', label=source['assoc name'])



###########################
###    データ整形         ###
###########################
obsdates_sed=np.round(np.unique(sed_tab['tstart'].data).tolist(),3)
obsdates_lc=np.round(lc_tab['tmin_mjd'],3)
array_validseds = []
for idx, obsdate_lc in enumerate(obsdates_lc):
  # print(idx, ': obsdate',obsdate_lc)
  mask = (obsdates_sed==obsdate_lc)
  if mask.sum() > 0:
    # print('obsdate', obsdate_lc, '---', mask.sum() )
    # print(lc_tab[idx])
    array_validseds.append(True)
  else:
    # print('obsdate', obsdate_lc)
    array_validseds.append(False)
from astropy.table import Column
lc_tab.add_column(array_validseds, name='ValidSED', index=0)
print(lc_tab)



###########################
###    描画              ###
###########################
### Light Curve ###
lc_x=lc_tab['t_mjd']
lc_y=lc_tab['flux']
plt.plot(lc_x,lc_y ,marker='o',label="Full Light Curve")

mask = (lc_tab['ValidSED'] == True)
lc_x=lc_tab[mask]['t_mjd']
lc_y=lc_tab[mask]['flux']
plt.plot(lc_x,lc_y ,marker='o', linestyle='', label="Valid SEDs")


plt.xlabel('MJD')
plt.ylabel('flux (cm$^{-2}$ s$^{-1}$)')
plt.yscale('log')
plt.ylim(1e-8, 1e-4)
# plt.xlim(57790-14*20-5, 57790+14*20+5)  # Adjust the x-axis limits to show a 14-day window around MJD 57790
plt.legend()
plt.show()
plt.tight_layout()

### Histogram of fluxes ###
hist_valid = plt.hist(np.log10(lc_tab[mask]['flux'].data), bins=np.arange(-6.5, -4.0, 0.1),histtype='step', lw=2, label='Valid SED')
hist_valid = plt.hist(np.log10(lc_tab[np.invert(mask)]['flux'].data), bins=np.arange(-6.5, -4.0, 0.1),histtype='step', lw=2, label='Invalid SED')
plt.legend()
print(lc_tab[mask]['flux'][0:10])
print(np.log10(lc_tab[mask]['flux'][0:10]))



In [ ]:
print(pow(10,-5.8))
print(pow(10,-5.9))




## Threshold 適用

In [ ]:
from astropy.table import Table
import matplotlib.pyplot as plt
import os

threshold_flux = 1.0e-6  # Define a threshold for flux
lc_file_dir = 'data_LC/14d'
file_list = get_lc_files(lc_file_dir) 

for file in file_list:
    if file.endswith('.fits'):
        lc_tab = Table.read(os.path.join(lc_file_dir, file))
        # Add a new column with the average of tmax_mjd and tmin_mjd
        lc_tab.add_column((lc_tab['tmax_mjd'] + lc_tab['tmin_mjd']) / 2., name='t_mjd', index=0)
        mask = (lc_tab['flux'] > threshold_flux) 
        # lc_tab.add_column((mask, name='high state', index=0)

        # fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(15,10))
        sourcename_4fgl = file.split('_')[1] # Extract the 4FGL name from the filename
        sourceinfo = dict_sourcename[dict_sourcename['4FGL name'] == '4fgl_' + sourcename_4fgl]
        sourcename_assoc = sourceinfo['assoc name'].data[0]
        sourceindex = sourceinfo['index'].data[0]
        print('4fgl_' + sourcename_4fgl + ':' + str(sourcename_assoc) + ' ' + str(sourceindex))
        print(np.round(lc_tab['t_mjd'][mask],1))
        # print(sourcename_assoc)
        lc_x=lc_tab['t_mjd']
        lc_y=lc_tab['flux']

        fig = plt.figure(figsize=(8, 5))
        plt.plot(lc_x,lc_y ,marker='o',label=f"{sourcename_assoc}, {sourceindex}")
        plt.plot(lc_x[mask], lc_y[mask], marker='o', linestyle='', color='red', label='High State')

        # lightcurve,= plt.plot(lc_x[fermipresentbin],lc_y[fermipresentbin] ,marker='o',color='red')
        # plt.set(xlim=[mjd_min, mjd_max], ylim=[0,0.3e-4])
        plt.xlabel('MJD')
        plt.ylabel('flux (cm$^{-2}$ s$^{-1}$)')
        plt.yscale('log')
        plt.ylim(1e-8, 1e-4)
        plt.legend()
        # plt.show()
        plt.tight_layout()
        plt.savefig(f"figures/lightcurve_{str(sourceindex)}_{str(sourcename_assoc)}.png", dpi=300, bbox_inches='tight')


        

